# Dataset Merger
### Economic Development by Political Indicators

Merges 6 datasets into one master CSV keyed on **`country_code` (ISO3) + `year`**

| # | Dataset | Source | Key Indicators |
|---|---------|--------|----------------|
| 1 | WGI | World Bank | Rule of Law, Corruption, Political Stability, Gov't Effectiveness, Voice & Accountability, Regulatory Quality |
| 2 | WEO | IMF | GDP Growth, Inflation, Unemployment, Govt Debt, GDP (USD) |
| 3 | Polity5 | Center for Systemic Peace | Democracy/Autocracy Score, Executive Constraints |
| 4 | V-Dem | Univ. of Gothenburg | Electoral, Liberal, Participatory Democracy Index |
| 5 | FIW | Freedom House | Political Rights, Civil Liberties, Free/PF/NF Status |
| 6 | HDI | UNDP | Human Development Index, GNI per capita, Life Expectancy |

**Output:** `Master_EconDev_PoliticalIndicators.csv`

---
## 0. Setup

In [1]:
import pandas as pd
import numpy as np
import pycountry
import os

FOLDER = "/Users/kongsattha/Documents/ITC/Data set/Economic_Political_Indicators"
os.chdir(FOLDER)

# Helper: country name → ISO3 code
MANUAL_ISO3 = {
    "Congo (Brazzaville)":            "COG",
    "Congo (Kinshasa)":               "COD",
    "St. Kitts and Nevis":            "KNA",
    "St. Lucia":                      "LCA",
    "St. Vincent and the Grenadines": "VCT",
    "Turkey":                         "TUR",
    "Burma":                          "MMR",
    "Ivory Coast":                    "CIV",
    "Laos":                           "LAO",
    "Kosovo":                         "XKX",
    "East Timor":                     "TLS",
    "Cape Verde":                     "CPV",
    "Swaziland":                      "SWZ",
    "Eswatini":                       "SWZ",
}

def name_to_iso3(name):
    if name in MANUAL_ISO3:
        return MANUAL_ISO3[name]
    try:
        return pycountry.countries.search_fuzzy(name)[0].alpha_3
    except:
        return np.nan

print("Setup complete. Working directory:", os.getcwd())

Setup complete. Working directory: /Users/kongsattha/Documents/ITC/Data set/Economic_Political_Indicators


---
## 1. World Bank — Worldwide Governance Indicators (WGI)

6 governance dimensions scored **-2.5 to +2.5** for 200+ countries, 1996–2024.  
Files are in **wide format** (one row per country, one column per year) — we melt them to long format.

In [2]:
wgi_files = {
    "wgi_voice_accountability":  "WGI_Voice_Accountability.csv",
    "wgi_political_stability":   "WGI_Political_Stability.csv",
    "wgi_govt_effectiveness":    "WGI_Government_Effectiveness.csv",
    "wgi_regulatory_quality":    "WGI_Regulatory_Quality.csv",
    "wgi_rule_of_law":           "WGI_Rule_of_Law.csv",
    "wgi_control_of_corruption": "WGI_Control_of_Corruption.csv",
}

wgi_dfs = []
for col_name, fname in wgi_files.items():
    df = pd.read_csv(fname, skiprows=4)
    year_cols = [c for c in df.columns if c != "Country Code"]
    df_long = df.melt(id_vars=["Country Code"], value_vars=year_cols,
                      var_name="year", value_name=col_name)
    df_long["year"] = pd.to_numeric(df_long["year"], errors="coerce")
    df_long = df_long.dropna(subset=["year"])
    df_long["year"] = df_long["year"].astype(int)
    df_long = df_long.rename(columns={"Country Code": "country_code"})
    wgi_dfs.append(df_long)

wgi = wgi_dfs[0]
for d in wgi_dfs[1:]:
    wgi = wgi.merge(d, on=["country_code", "year"], how="outer")

wgi = wgi[wgi["country_code"].str.match(r"^[A-Z]{3}$", na=False)]
wgi = wgi.dropna(subset=list(wgi_files.keys()), how="all")

print(f"WGI: {wgi.shape[0]:,} rows | {wgi['country_code'].nunique()} countries | {wgi['year'].min()}–{wgi['year'].max()}")
wgi.head()

WGI: 5,083 rows | 205 countries | 1996–2023


,country_code,year,wgi_voice_accountability,wgi_political_stability,wgi_govt_effectiveness,wgi_regulatory_quality,wgi_rule_of_law,wgi_control_of_corruption
44,ABW,2004,0.737521,0.993218,1.281389,0.762954,0.915527,1.165965
45,ABW,2005,1.15323,1.376416,1.287598,0.864074,0.847438,1.269663
46,ABW,2006,1.017273,1.303461,1.279818,0.855287,0.810431,1.251313
47,ABW,2007,0.972129,1.286569,1.246601,0.877206,0.826464,1.261871
48,ABW,2008,0.967198,1.316396,1.270925,0.894702,0.847226,1.276775


---
## 2. IMF — World Economic Outlook (WEO)

Macroeconomic indicators for 229 countries, already in **long format** (country + year rows).

In [3]:
imf = pd.read_csv("IMF_WEO_2024.csv")
imf = imf.rename(columns={
    "country":                        "country_code",
    "GDP_Growth_Rate_pct":            "imf_gdp_growth_pct",
    "Inflation_Rate_pct":             "imf_inflation_pct",
    "Unemployment_Rate_pct":          "imf_unemployment_pct",
    "Current_Account_Balance_USD_bn": "imf_current_account_usd_bn",
    "Gross_Govt_Debt_pct_GDP":        "imf_govt_debt_pct_gdp",
    "GDP_Current_Prices_USD_bn":      "imf_gdp_usd_bn",
})
imf["year"] = pd.to_numeric(imf["year"], errors="coerce").astype("Int64")
imf = imf[imf["country_code"].str.match(r"^[A-Z]{3}$", na=False)]

print(f"IMF: {imf.shape[0]:,} rows | {imf['country_code'].nunique()} countries | {imf['year'].min()}–{imf['year'].max()}")
imf.head()

IMF: 10,431 rows | 220 countries | 1980–2030


,country_code,year,imf_gdp_growth_pct,imf_inflation_pct,imf_unemployment_pct,imf_current_account_usd_bn,imf_govt_debt_pct_gdp,imf_gdp_usd_bn
0,SDN,1980,2.5,26.5,NaN,-0.931,NaN,9.095
1,SDN,1981,6.3,24.0,22.3,-1.371,NaN,6.530
2,SDN,1982,4.1,26.6,22.2,-1.100,NaN,4.748
3,SDN,1983,-1.5,30.6,21.4,-0.713,NaN,6.485
4,SDN,1984,-5.6,33.7,20.7,-0.766,NaN,7.992


---
## 3. Polity5 — Regime Type

Democracy/autocracy scores from **-10 (full autocracy)** to **+10 (full democracy)** for 194 countries, 1800–2018.  
Special codes (-66, -77, -88) represent interruptions/transitions — replaced with `NaN`.

In [4]:
polity = pd.read_csv("Polity5.csv")
polity = polity.rename(columns={"scode": "country_code"})
polity["year"] = pd.to_numeric(polity["year"], errors="coerce").astype("Int64")

polity = polity[["country_code", "year", "democ", "autoc", "polity", "polity2",
                 "durable", "xconst", "parcomp"]].copy()
polity.columns = ["country_code", "year",
                  "polity_democ", "polity_autoc", "polity_score", "polity2_score",
                  "polity_durable", "polity_exec_constraints", "polity_party_competition"]

# Replace special transition codes with NaN
for c in ["polity_democ", "polity_autoc", "polity_score", "polity2_score"]:
    polity[c] = polity[c].where(polity[c] >= -10, np.nan)

polity = polity[polity["country_code"].str.match(r"^[A-Z]{2,4}$", na=False)]

print(f"Polity5: {polity.shape[0]:,} rows | {polity['country_code'].nunique()} countries | {polity['year'].min()}–{polity['year'].max()}")
polity.head()

Polity5: 17,574 rows | 194 countries | 1776–2020


,country_code,year,polity_democ,polity_autoc,polity_score,polity2_score,polity_durable,polity_exec_constraints,polity_party_competition
0,AFG,1800,1.0,7.0,-6.0,-6.0,NaN,1,3
1,AFG,1801,1.0,7.0,-6.0,-6.0,NaN,1,3
2,AFG,1802,1.0,7.0,-6.0,-6.0,NaN,1,3
3,AFG,1803,1.0,7.0,-6.0,-6.0,NaN,1,3
4,AFG,1804,1.0,7.0,-6.0,-6.0,NaN,1,3


---
## 4. V-Dem — Varieties of Democracy

Three core democracy indices (0–1 scale) for 176 countries, 1789–2025.

In [5]:
vdem = pd.read_csv("VDem_Core_Indices.csv")
vdem = vdem.rename(columns={
    "Code":                          "country_code",
    "Year":                          "year",
    "Electoral_Democracy_Index":     "vdem_electoral_democracy",
    "Liberal_Democracy_Index":       "vdem_liberal_democracy",
    "Participatory_Democracy_Index": "vdem_participatory_democracy",
})
vdem = vdem[["country_code", "year", "vdem_electoral_democracy",
             "vdem_liberal_democracy", "vdem_participatory_democracy"]]
vdem["year"] = pd.to_numeric(vdem["year"], errors="coerce").astype("Int64")
vdem = vdem[vdem["country_code"].str.match(r"^[A-Z]{3}$", na=False)]

print(f"V-Dem: {vdem.shape[0]:,} rows | {vdem['country_code'].nunique()} countries | {vdem['year'].min()}–{vdem['year'].max()}")
vdem.head()

V-Dem: 29,649 rows | 176 countries | 1789–2025


,country_code,year,vdem_electoral_democracy,vdem_liberal_democracy,vdem_participatory_democracy
0,AFG,1789,0.019,0.029,0.018
1,AFG,1790,0.019,0.029,0.022
2,AFG,1791,0.019,0.029,0.022
3,AFG,1792,0.019,0.029,0.022
4,AFG,1793,0.019,0.029,0.022


---
## 5. Freedom House — Freedom in the World (FIW)

Annual political rights and civil liberties scores for 195 countries, 2013–2025.  
Country names are mapped to ISO3 codes using `pycountry` + a manual override dictionary.

In [6]:
fh = pd.read_csv("Freedom_House_FIW_2013-2025.csv")
fh = fh.rename(columns={
    "Country/Territory": "country_name",
    "Edition":           "year",
    "Status":            "fh_status",
    "PR rating":         "fh_political_rights_rating",
    "CL rating":         "fh_civil_liberties_rating",
    "PR":                "fh_political_rights_score",
    "CL":                "fh_civil_liberties_score",
    "Total":             "fh_total_score",
})
fh = fh[fh["C/T"] == "c"]   # countries only, exclude territories
fh = fh[["country_name", "year", "fh_status", "fh_political_rights_rating",
          "fh_civil_liberties_rating", "fh_political_rights_score",
          "fh_civil_liberties_score", "fh_total_score"]].copy()
fh["year"] = pd.to_numeric(fh["year"], errors="coerce").astype("Int64")
fh["country_code"] = fh["country_name"].apply(name_to_iso3)
fh = fh.dropna(subset=["country_code"])

print(f"Freedom House: {fh.shape[0]:,} rows | {fh['country_code'].nunique()} countries | {fh['year'].min()}–{fh['year'].max()}")
fh.head()

Freedom House: 2,535 rows | 194 countries | 2013–2025


,country_name,year,fh_status,fh_political_rights_rating,fh_civil_liberties_rating,fh_political_rights_score,fh_civil_liberties_score,fh_total_score,country_code
1,Afghanistan,2025,NF,7,7,1,5,6,AFG
2,Albania,2025,PF,3,3,28,40,68,ALB
3,Algeria,2025,NF,6,5,10,21,31,DZA
4,Andorra,2025,F,1,1,38,55,93,AND
5,Angola,2025,NF,6,5,10,18,28,AGO


---
## 6. UNDP — Human Development Index (HDI)

Cross-sectional data for 2022 (published in the 2023–24 HDR report).  
Read directly from the Excel file — header rows are auto-detected.

In [7]:
hdi_raw = pd.read_excel("HDI_2023-24.xlsx", header=None)

# Find the first data row (where column 0 == 1, i.e. rank #1)
data_start = next(
    (i for i, row in hdi_raw.iterrows()
     if str(row.iloc[0]).strip() == "1.0" or str(row.iloc[0]).strip() == "1"),
    5
)

hdi_data = hdi_raw.iloc[data_start:].copy().reset_index(drop=True)
hdi_data.columns = range(hdi_data.shape[1])

# Column layout: 0=rank, 1=country, 2=HDI, 4=life_expectancy,
#                6=expected_schooling, 8=mean_schooling, 10=GNI_per_capita
hdi = hdi_data[[0, 1, 2, 4, 6, 8, 10]].copy()
hdi.columns = ["hdi_rank", "country_name", "hdi_value", "hdi_life_expectancy",
               "hdi_expected_schooling_yrs", "hdi_mean_schooling_yrs", "hdi_gni_per_capita_ppp"]

hdi = hdi.dropna(subset=["country_name"])
hdi = hdi[hdi["country_name"].astype(str).str.strip().str.len() > 1]
hdi = hdi[~hdi["country_name"].astype(str).str.isupper()]  # drop group headers
hdi["year"] = 2022

for c in ["hdi_value", "hdi_life_expectancy", "hdi_mean_schooling_yrs",
          "hdi_expected_schooling_yrs", "hdi_gni_per_capita_ppp"]:
    hdi[c] = pd.to_numeric(hdi[c], errors="coerce")

hdi["country_code"] = hdi["country_name"].astype(str).apply(name_to_iso3)
hdi = hdi.dropna(subset=["country_code", "hdi_value"])

print(f"HDI: {hdi.shape[0]:,} rows | {hdi['country_code'].nunique()} countries | year 2022 (cross-sectional)")
hdi.head()

HDI: 183 rows | 182 countries | year 2022 (cross-sectional)


,hdi_rank,country_name,hdi_value,hdi_life_expectancy,hdi_expected_schooling_yrs,hdi_mean_schooling_yrs,hdi_gni_per_capita_ppp,year,country_code
0,1,Switzerland,0.967,84.255,16.583731,13.904066,69432.78669,2022,CHE
1,2,Norway,0.966,83.393,18.638460,13.062343,69189.76165,2022,NOR
2,3,Iceland,0.959,82.815,19.106730,13.767170,54688.37921,2022,ISL
4,5,Denmark,0.952,81.882,18.774031,12.960490,62018.95694,2022,DNK
5,5,Sweden,0.952,83.505,19.036770,12.673720,56995.84796,2022,SWE


---
## 7. Merge All Datasets

Merge strategy:
- **Base:** WGI (best country+year coverage 1996–2024)
- **Outer join** with IMF, V-Dem, Freedom House, Polity5 → keep all observations
- **Left join** with HDI → only fills in 2022 rows (cross-sectional)
- **Filter** to 1996–2024 where data overlap is greatest

In [8]:
master = wgi.copy()
master["year"] = master["year"].astype("Int64")

master = master.merge(imf,  on=["country_code", "year"], how="outer")
master = master.merge(vdem, on=["country_code", "year"], how="outer")
master = master.merge(
    fh[["country_code", "year", "fh_status", "fh_political_rights_rating",
        "fh_civil_liberties_rating", "fh_political_rights_score",
        "fh_civil_liberties_score", "fh_total_score"]],
    on=["country_code", "year"], how="outer"
)
master = master.merge(polity, on=["country_code", "year"], how="outer")
master = master.merge(
    hdi[["country_code", "year", "hdi_value", "hdi_life_expectancy",
         "hdi_mean_schooling_yrs", "hdi_expected_schooling_yrs", "hdi_gni_per_capita_ppp"]],
    on=["country_code", "year"], how="left"
)

master = master.sort_values(["country_code", "year"]).reset_index(drop=True)
master = master[(master["year"] >= 1996) & (master["year"] <= 2024)].copy()

print(f"Master dataset: {master.shape[0]:,} rows × {master.shape[1]} columns")
print(f"Countries: {master['country_code'].nunique()}")
print(f"Years: {master['year'].min()} – {master['year'].max()}")

Master dataset: 8,703 rows × 35 columns
Countries: 324
Years: 1996 – 2024


---
## 8. Coverage Report

In [9]:
total = len(master)
coverage = pd.DataFrame({
    "non_null": master.notna().sum(),
    "coverage_pct": (master.notna().sum() / total * 100).round(1)
})
coverage = coverage[~coverage.index.isin(["country_code", "year"])].sort_values("coverage_pct", ascending=False)
coverage

,non_null,coverage_pct
imf_gdp_usd_bn,6282,72.2
imf_gdp_growth_pct,6259,71.9
imf_inflation_pct,6200,71.2
imf_current_account_usd_bn,6082,69.9
imf_govt_debt_pct_gdp,5840,67.1
wgi_rule_of_law,5091,58.5
wgi_voice_accountability,5064,58.2
vdem_electoral_democracy,5058,58.1
vdem_liberal_democracy,5049,58.0
vdem_participatory_democracy,5052,58.0


---
## 9. Preview Master Dataset

In [10]:
master.head(10)

,country_code,year,wgi_voice_accountability,wgi_political_stability,wgi_govt_effectiveness,wgi_regulatory_quality,wgi_rule_of_law,wgi_control_of_corruption,imf_gdp_growth_pct,imf_inflation_pct,...,polity_score,polity2_score,polity_durable,polity_exec_constraints,polity_party_competition,hdi_value,hdi_life_expectancy,hdi_mean_schooling_yrs,hdi_expected_schooling_yrs,hdi_gni_per_capita_ppp
10,ABW,1996,NaN,NaN,NaN,NaN,NaN,NaN,1.3,3.2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
11,ABW,1997,NaN,NaN,NaN,NaN,NaN,NaN,7.0,3.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
12,ABW,1998,NaN,NaN,NaN,NaN,NaN,NaN,2.0,1.9,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
13,ABW,1999,NaN,NaN,NaN,NaN,NaN,NaN,1.4,2.3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
14,ABW,2000,NaN,NaN,NaN,NaN,NaN,NaN,7.7,4.1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
15,ABW,2001,NaN,NaN,NaN,NaN,NaN,NaN,4.2,2.9,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
16,ABW,2002,NaN,NaN,NaN,NaN,NaN,NaN,-0.9,3.3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
17,ABW,2003,NaN,NaN,NaN,NaN,NaN,NaN,1.1,3.7,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
18,ABW,2004,0.737521,0.993218,1.281389,0.762954,0.915527,1.165965,7.3,2.5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
19,ABW,2005,1.15323,1.376416,1.287598,0.864074,0.847438,1.269663,-0.4,3.4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [11]:
master.describe().round(3)

,year,imf_gdp_growth_pct,imf_inflation_pct,imf_unemployment_pct,imf_current_account_usd_bn,imf_govt_debt_pct_gdp,imf_gdp_usd_bn,vdem_electoral_democracy,vdem_liberal_democracy,vdem_participatory_democracy,...,polity_score,polity2_score,polity_durable,polity_exec_constraints,polity_party_competition,hdi_value,hdi_life_expectancy,hdi_mean_schooling_yrs,hdi_expected_schooling_yrs,hdi_gni_per_capita_ppp
count,8703.0,6259.000,6200.000,3250.000,6082.000,5840.000,6282.000,5058.000,5049.000,5052.000,...,3652.000,3748.000,3797.000,3802.000,3802.000,185.000,185.000,185.000,185.000,185.000
mean,2009.397,3.528,23.374,8.862,1.838,55.161,1092.074,0.512,0.399,0.329,...,3.625,3.549,24.445,1.811,0.253,0.722,71.783,8.953,13.472,21102.262
std,8.061,5.904,870.622,6.008,104.479,42.424,4190.000,0.264,0.269,0.205,...,6.462,6.401,28.837,15.843,15.456,0.157,7.911,3.289,3.071,22612.493
min,1996.0,-54.300,-72.700,0.700,-1259.631,0.000,0.014,0.014,0.005,0.008,...,-10.000,-10.000,0.000,-88.000,-88.000,0.380,52.997,1.341,5.635,690.661
25%,2003.0,1.500,1.800,4.900,-1.726,29.500,6.151,0.276,0.148,0.141,...,-2.000,-2.000,5.000,3.000,2.000,0.601,65.694,6.432,11.531,4754.836
50%,2009.0,3.700,3.800,7.300,-0.186,46.600,30.962,0.514,0.366,0.316,...,6.000,6.000,15.000,5.000,4.000,0.740,72.300,9.248,13.242,12360.816
75%,2016.0,5.800,7.725,10.900,1.158,69.400,279.812,0.765,0.642,0.492,...,9.000,9.000,32.000,7.000,4.000,0.847,77.905,11.613,15.662,32171.246
max,2024.0,148.000,65374.100,60.800,903.623,600.100,49422.823,0.922,0.896,0.808,...,10.000,10.000,170.000,7.000,5.000,0.967,84.820,14.256,21.080,146673.242


---
## 10. Save Master Dataset

In [12]:
out_path = f"{FOLDER}/Master_EconDev_PoliticalIndicators.csv"
master.to_csv(out_path, index=False)

size_mb = os.path.getsize(out_path) / 1024 / 1024
print(f"Saved: {out_path}")
print(f"Size:  {size_mb:.2f} MB")
print(f"Shape: {master.shape[0]:,} rows × {master.shape[1]} columns")

Saved: /Users/kongsattha/Documents/ITC/Data set/Economic_Political_Indicators/Master_EconDev_PoliticalIndicators.csv
Size:  1.19 MB
Shape: 8,703 rows × 35 columns
